In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 从项目根目录或 examples 目录启动均可；统一以项目根目录运行。
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "mtp_initializer").is_dir() and (p / "PROJECT.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 scipykit 项目根目录或 examples 目录启动 notebook")
os.chdir(PROJECT_ROOT)
# 根目录用于本地脚本；父目录用于 import scipykit。同步对子进程生效。
python_paths = [str(PROJECT_ROOT), str(PROJECT_ROOT.parent)]
for path in reversed(python_paths):
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(python_paths + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
)
os.environ["NOTEBOOK_NAME"] = "05_数据与文件IO"
print("项目路径:", PROJECT_ROOT)
print("当前解释器:", sys.executable)


# 数据与基础文件 IO
`sci_initializer` 是数据/文件工具入口，`task.file` 保留原来的组合导入。此示例展示中文 JSON、可信本地 pickle、Markdown 表格以及 DataFrame 外置保存。

In [ ]:
from scipykit.sci_initializer import *
from scipykit.mtp_initializer import disp
root = Path("assets/05_数据与文件IO")
root.mkdir(parents=True, exist_ok=True)
config = {"实验": "exp01", "随机种子": 42, "batch_size": 16}
json_save(config, root / "config.json", indent=2)
assert json_load(root / "config.json") == config
# pickle 只用于自己创建或可信来源的本地文件。
pkl_save(np.arange(6).reshape(2, 3), root / "array.pkl")
print(pkl_load(root / "array.pkl"))

In [ ]:
frame = pd_read_markdown_table("""
| 方法 | 准确率 | 备注 |
| --- | --- | --- |
| **基线** | 0.82 | |
| 改进 | 0.91 | `实验一` |
""")
disp(frame)
paths = disp(frame, "metrics", df_formats=("csv", "xlsx"))
print("已保存:", paths)
# 省略 df_formats 时优先 Parquet；没有引擎时自动选择 CSV。
auto_paths = disp(frame, "auto_metrics", df_link_html=False)
print("自动格式:", auto_paths)
assert "parquet" in auto_paths
pd.testing.assert_frame_equal(pd.read_parquet(auto_paths["parquet"]), frame)
assert all(path.is_absolute() for path in auto_paths.values())

安装 pyarrow 或 fastparquet 后默认保存 Parquet，未安装时自动回退 CSV。本示例包含 Parquet 与 Excel 验证，需准备 pyarrow 和 openpyxl；也可显式 `df_formats=("parquet", "csv", "xlsx")` 同时导出。`disp(df)` 默认仅紧凑纯文本；`df_render_html=True` 让未命名表格内嵌 HTML，命名表格的完整 HTML 位于外部资源目录。